# KALIA v0.3.0 Step 1 — the 58M control seed spread

**This is not an experiment. It has no hypothesis and no threshold.** It measures
the one number the project never had: how much KALIA's own control moves when
nothing about it changes.

At 30M the answer was **~0.11 nats** (X21: seed 1337 gives 4.7662 from two
independent sessions, 0.0018 apart; seeds 1338/1339 give 4.8769/4.8908). Two
sessions agreeing to 0.0018 means the machine is not the variable, so that gap
is a seed effect. **We do not know the 58M figure.**

Every arm in X18–X20 was compared against an unmeasured baseline, which is why
X18's −0.0436 and X19's −0.0018 are now nulls and X20's −0.1358 inverted.
This step costs about a sixth of the monthly quota to not repeat that.

**Registered stopping rule: if the spread exceeds 0.15 nats, stop and
re-derive every threshold in the pre-registration.** At that noise level this
project cannot measure 58M changes at a micro budget, and continuing would only
manufacture more X20s.

In [ ]:
# Registered constants, read from the pre-registration at build time.
SEEDS_HERE = [1401, 1402, 1403]
STOP_SPREAD_HERE = 0.15
print('registered seeds:', SEEDS_HERE, '| stopping threshold:', STOP_SPREAD_HERE)

In [ ]:
import glob, json, os, shutil, subprocess, sys

work = '/kaggle/working/kalia'
if not os.path.exists(work):
    hits = sorted(glob.glob('/kaggle/input/**/train.py', recursive=True))
    assert hits, 'attach the kalia-code-dev dataset'
    shutil.copytree(os.path.dirname(hits[0]), work)
os.chdir(work)
data = sorted(glob.glob('/kaggle/input/**/train.bin', recursive=True))
assert data, 'train.bin not found - attach a prep output as a kernel source'
DATA = os.path.dirname(data[0])
print('code:', work)
print('data:', DATA)
import torch
print('gpu:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('adaptive_every supported:', 'adaptive_every' in open('train.py').read())

In [ ]:
SEEDS = list(SEEDS_HERE)
arms = [f'v030-ctl-s{s}' for s in SEEDS]
for a in arms:
    p = f'configs/{a}.yaml'
    assert os.path.exists(p), f'missing {p}'
    txt = open(p).read()
    print(f'  {a:18s} seed={txt.split("seed:")[1].split()[0]}')
print('\nThese configs must differ from kalia-v020.yaml only in the seed.')
print('That is asserted by test_kautilya_config.py for the Kautilya arm.')

In [ ]:
print(subprocess.run([sys.executable, 'ablate.py',
                       '--arms', ','.join(arms),
                       '--steps', '500',
                       '--data-dir', DATA,
                       '--out-root', '/kaggle/working/out'],
                      capture_output=True, text=True).stdout[-5000:])

In [ ]:
import pandas as pd
losses = {}
print('=== control arms ===')
for a in arms:
    df = pd.read_csv(f'/kaggle/working/out/{a}/val_log.csv')
    line = '  '.join(f"{int(r.step)}={r.val_loss:.4f}" for r in df.itertuples())
    print(f'{a:18s} {line}')
    losses[a] = float(df.iloc[-1].val_loss)
vals = [losses[a] for a in arms]
spread = max(vals) - min(vals)
print(f'\n58M control seed spread: {min(vals):.4f} .. {max(vals):.4f} = {spread:.4f} nats')
print(f'registered stopping threshold: {STOP_SPREAD_HERE}')
STOPPING = spread > STOP_SPREAD_HERE
print('STOP THE PROGRAMME' if STOPPING else 'within budget -- thresholds are 1x this spread')
res = {'losses': losses, 'spread': spread, 'stopping': STOPPING,
       'seeds': SEEDS_HERE, 'bar': spread}
json.dump(res, open('/kaggle/working/step1_spread.json', 'w'), indent=2)

In [ ]:
print('=== comparison with the 30M measurement ===')
res = json.load(open('/kaggle/working/step1_spread.json'))
print(f"  30M (X21): spread 0.1139 nats -- larger than X18's entire -0.0436")
print(f"  58M (here): spread {res['spread']:.4f} nats")
if res['stopping']:
    print('\n  Registered stopping rule triggered. Do NOT run Steps 2-4.')
    print('  Either spend more compute per arm, or stop publishing')
    print('  micro-scale claims at 58M entirely.')
else:
    print('\n  Every threshold in Steps 2-4 is now 1x this number.')